### Qwote: Training data generation

Set up environment (including `assume --env` credentials if required)

In [ ]:
import os, shutil
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datagen import BedrockBatchGenerator, TrainingDataUploader, DocumentLoader, BatchOutputCorrector
from opnoteschema.prompt_builder import PromptBuilder
from opnoteschema import Schema

In [ ]:
pb = PromptBuilder()

#### Download documents from batch

In [ ]:
DocumentLoader.list_available_document_batches()

In [ ]:
gen = BedrockBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    schema=Schema,
    schema_name="opnoteschema",
    model_name="sonnet4",
    document_batches=["opnotes_sonnet45_v2-2026-09-25-001.tar.gz", "normal-batch-2025-12-31-001.tar.gz"],
)

In [ ]:
gen.get_document_files_count()

#### Sanity check

##### Start batch generation

In [ ]:
gen.generate_via_batch(100, os.environ["BUCKET"],os.environ["BEDROCK_EXECUTION_ROLE"])

##### Download and parse batch outputs

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Start datagen

In [ ]:
gen.generate_via_batch(3000, os.environ["BUCKET"], os.environ["BEDROCK_EXECUTION_ROLE"])

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Correct batch outputs

In [ ]:
corrector = BatchOutputCorrector(Schema, Path("anthropic_batch_job.jsonl.out"))

In [ ]:
corrector.correct_with_claude(Path("prompts/correction.txt"), pb.build_datagen_prompt())

In [ ]:
corrector.correct_with_rules(Path("rules/correction.yaml"))

##### Re-extract corrected outputs

In [ ]:
shutil.rmtree("data/trainingdata")
gen.extract_batch_output()

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=Schema,
    schema_name="opnoteschema",
    system_prompt=pb.build_main_prompt(),
    short_description="qwote-batch",
    long_description="Training data for Qwote",
    input_folder=Path("./data/trainingdata"),
)